---

## **Problem Statement**

While basic AI agents can respond to individual queries effectively, they lack the ability to **retain conversational context**. This becomes a limitation in real-world scenarios where users refer to previous messages or ask follow-up questions.

We aim to enhance our LangChain-based AI agent by integrating **Conversation Memory**, allowing it to recall previous user inputs and its own responses within a session.

---

## **Objectives**

1. Use the `ConversationBufferMemory` module from LangChain to maintain session context.
2. Modify the LangChain agent setup to support memory.
3. Ensure compatibility with `create_tool_calling_agent()` and the latest LangChain version.
4. Maintain clean and modular code, with all required components defined clearly.
5. Ensure the agent provides more coherent, context-aware responses in multi-turn conversations.

---

## **Expected Outcomes**

- The agent will remember what was said previously in the session.
- It will respond appropriately to follow-up or context-based queries.
- It will behave more like a real assistant, capable of multi-turn conversation.
- All code will use **latest APIs and avoid deprecation warnings**.

---

In [ ]:
# Step1: Imports

from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from dotenv import load_dotenv
import os

from langchain.memory import ConversationBufferMemory
# ConversationBufferMemory is a memory implementation in LangChain that:
# -Stores full conversation history (user and AI messages).
# -Maintains order and context across multiple turns.
# -Can be plugged into agents and chains so the LLM remembers what happened earlier in the conversation.
# -It is the most basic type of memory in LangChain.

# Other types of memory include:
# - ConversationSummaryMemory: Summarizes conversation history to keep context concise.
# - ConversationSummaryBufferMemory: Combines summary and buffer memory.

/Users/ingledarshan/.local/share/virtualenvs/AH-KbrtVpZi/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Step 2: Load API key
load_dotenv()
# Load environment variables from a .env file, allowing for secure configuration management.

True

In [ ]:
# Additional imports for the tool
import psutil
# psutil - A cross-platform library for retrieving information on running processes and system utilization (CPU, memory, disks, network, sensors).
from datetime import datetime
# datetime - A module for manipulating dates and times, useful for logging and timestamps.
import platform
# platform - A module that provides information about the underlying platform (OS, Python version, etc.).

In [ ]:
# Step 3: Define calculator tool
@tool # Marks this function as a LangChain-compatible tool. This makes the calculator() function callable by the agent when it decides a mathematical expression should be evaluated.
def calculator(expression: str) -> float:
    """Evaluate a math expression like '45 * 2 + 100 / 4'.""" # Docstring: A natural language explanation of what this tool does. Think of this as documentation for the agent.
    return eval(expression)

@tool # Marks this function as a LangChain-compatible tool. This makes the battery_percentage() function callable by the agent when it decides to check the battery status.
def battery_percentage() -> str:
    """Get the current battery percentage of the laptop."""
    battery = psutil.sensors_battery()
    if battery is None:
        return "Battery information is not available."
    if battery.percent > 80:
        return f"Battery is at {battery.percent}%, which is good."
    elif battery.percent > 60:
        return f"Battery is at {battery.percent}%, which is okay."
    elif battery.percent > 40:
        return f"Battery is at {battery.percent}%, which is low."
    else:
        return f"Battery is at {battery.percent}%, which is critical."

@tool # Marks this function as a LangChain-compatible tool. This makes the current_time() function callable by the agent when it decides to check the current time.
def current_time() -> str:
    """Get the current time in HH:MM:SS format."""
    current_time = datetime.now()
    return f"Current time: {current_time.strftime('%H:%M:%S')}"

@tool # Marks this function as a LangChain-compatible tool. This makes the today() function callable by the agent when it decides to check today's date.
def today() -> str:
    """Get today's date."""
    today = datetime.now()
    return f"Today's date: {today.strftime('%Y-%m-%d')}"

@tool # Marks this function as a LangChain-compatible tool. This makes the system_info() function callable by the agent when it decides to check system information.
def system_info() -> str:
    """Get system name and version."""
    system_name = platform.system()
    system_version = platform.version()
    return f"System name: {system_name}, Version: {system_version}"

@tool # Marks this function as a LangChain-compatible tool. This makes the cpu_usage() function callable by the agent when it decides to check CPU usage.
def cpu_usage() -> str:
    """Get the current CPU usage percentage."""
    cpu_usage = psutil.cpu_percent(interval=1)
    return f"CPU usage: {cpu_usage}%"

@tool
def get_company_info(company: str) -> str:
    """Returns a short description of a given company."""
    company_data = {
        "oracle": "Oracle provides cloud infrastructure and enterprise software solutions.",
        "openai": "OpenAI builds advanced AI systems including ChatGPT and GPT models.",
        "google": "Google is known for its search engine, advertising, and cloud services.",
        "microsoft": "Microsoft develops software, hardware, and cloud services, including Windows and Azure.",
        "amazon": "Amazon is a global e-commerce and cloud computing giant, known for AWS and its online marketplace."
    }
    return company_data.get(company.lower(), "Sorry, I don't have information on that company.") # Oracle -> oracle, OpenAI -> openai, Google -> google

@tool
def calculate_tax(income: float) -> str:
    """Estimate tax based on annual income (India - simplified slab logic)."""
    if income <= 250000:
        tax = 0
    elif income <= 500000:
        tax = (income - 250000) * 0.05
    elif income <= 1000000:
        tax = (250000 * 0.05) + (income - 500000) * 0.2
    else:
        tax = (250000 * 0.05) + (500000 * 0.2) + (income - 1000000) * 0.3
    return f"Estimated tax: ₹{tax:.2f}"

@tool
def calculate_gst(base_price: float, gst_percent: float = 18.0) -> str:
    """Calculate final price including GST."""
    gst_amount = base_price * gst_percent / 100
    final_price = base_price + gst_amount
    return f"GST: ₹{gst_amount:.2f}, Final Price: ₹{final_price:.2f}"


tools = [calculator, battery_percentage, current_time, today, system_info, cpu_usage, get_company_info, calculate_tax, calculate_gst]

In [ ]:
# Step 4: Initialize the LLM
# https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.base.ChatOpenAI.html
llm = ChatOpenAI(model="gpt-4o", temperature=0.5)  # Using gpt-4o for better performance and tool calling capabilities

In [ ]:
# Step 5: Define memory to maintain conversation context during the agent's execution
memory = ConversationBufferMemory(
    memory_key="chat_history",  # Key under which the conversation history will be stored
    return_messages=True  # Instructs langchain to return the full message objects (including role and content) in the memory
)

# chat_history - It is not a keyword argument, but a memory_key argument. It is used to store the conversation history in the memory object. You can name it anything you want, but it is common to use "chat_history" or "history".

/var/folders/5g/9xpg7d6d4114s98tv10y9rgm0000gn/T/ipykernel_82200/3850363182.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [ ]:
# Step 5: Create prompt template (IMPORTANT: include input & agent_scratchpad)
# prompt = ChatPromptTemplate.from_messages([
#     ("system", "You are a smart assistant. You can perform calculations, check battery status, current time, today's date, system information, and CPU usage."),
#     ("user", "{input}"),
#     MessagesPlaceholder(variable_name="agent_scratchpad")  # Required
#     # MessagesPlaceholder - A special placeholder used by LangChain agents to store: a. Intermediate reasoning steps, b. Tool calls, c. Tool outputs - ReAct Style (Thoughts, Actions, and Observations)
# ])

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a smart assistant. You can perform calculations, check battery status, current time, today's date, system information, and CPU usage. Since you are storing the chat history for the conversation, every time you generate a response, start by clearly specify whether you referred the chat history for this response or not with 'Referring chat history' or 'Not referring chat history.' and then provide your response on the next line."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

In [ ]:
# Step 6: Create agent
agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)

In [ ]:
# Step 7: Wrap in executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, memory=memory)
# verbose=True - Shows detailed logs of how the agent reasons, chooses tools, and responds

In [ ]:
response = agent_executor.invoke({"input": "What is todays date?"})
print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `today` with `{}`


Today's date: 2026-03-05Not referring chat history.
Today's date is March 5, 2026.

> Finished chain.
Not referring chat history.
Today's date is March 5, 2026.


In [ ]:
response = agent_executor.invoke({"input": "What is the current time?"})
print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `current_time` with `{}`


Current time: 15:24:35Not referring chat history.
The current time is 15:24:35.

> Finished chain.
Not referring chat history.
The current time is 15:24:35.


In [ ]:
response = agent_executor.invoke({"input": "What is the current time in 12hr format?"})
print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `current_time` with `{}`


Current time: 15:25:16Not referring chat history.
The current time in 12-hour format is 03:25:16 PM.

> Finished chain.
Not referring chat history.
The current time in 12-hour format is 03:25:16 PM.


In [ ]:
response = agent_executor.invoke({"input": "What is my name?"})
# Pretty print the response
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Not referring chat history.
I'm sorry, but I don't have access to your name. Could you please tell me your name?

> Finished chain.
Response: Not referring chat history.
I'm sorry, but I don't have access to your name. Could you please tell me your name?


In [ ]:
response = agent_executor.invoke({"input": "What is my system information?"})
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...

Invoking: `system_info` with `{}`


System name: Darwin, Version: Darwin Kernel Version 25.3.0: Wed Jan 28 20:49:24 PST 2026; root:xnu-12377.81.4~5/RELEASE_ARM64_T8132Not referring chat history.
Your system information is as follows:  
System Name: Darwin  
Version: Darwin Kernel Version 25.3.0: Wed Jan 28 20:49:24 PST 2026; root:xnu-12377.81.4~5/RELEASE_ARM64_T8132

> Finished chain.
Response: Not referring chat history.
Your system information is as follows:  
System Name: Darwin  
Version: Darwin Kernel Version 25.3.0: Wed Jan 28 20:49:24 PST 2026; root:xnu-12377.81.4~5/RELEASE_ARM64_T8132


In [ ]:
response = agent_executor.invoke({"input": "What is my battery percentage?"})
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...

Invoking: `battery_percentage` with `{}`


Battery is at 40%, which is critical.Not referring chat history.
Your battery is currently at 40%, which is considered critical.

> Finished chain.
Response: Not referring chat history.
Your battery is currently at 40%, which is considered critical.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "Tell me about Oracle."})

# Step 9: Output the result
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...

Invoking: `get_company_info` with `{'company': 'Oracle'}`


Oracle provides cloud infrastructure and enterprise software solutions.Not referring chat history.
Oracle provides cloud infrastructure and enterprise software solutions.

> Finished chain.
Response: Not referring chat history.
Oracle provides cloud infrastructure and enterprise software solutions.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "Tell me about Walmart."})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...

Invoking: `get_company_info` with `{'company': 'Walmart'}`


Sorry, I don't have information on that company.Not referring chat history.
I'm sorry, but I don't have information on Walmart at the moment.

> Finished chain.
Response: Not referring chat history.
I'm sorry, but I don't have information on Walmart at the moment.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "Can you remind me what Oracle was again?"})

# Step 9: Output the result
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Referring chat history.
Oracle provides cloud infrastructure and enterprise software solutions.

> Finished chain.
Response: Referring chat history.
Oracle provides cloud infrastructure and enterprise software solutions.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "What does Oracle do?"})

# Step 9: Output the result
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Referring chat history.
Oracle provides cloud infrastructure and enterprise software solutions.

> Finished chain.
Response: Referring chat history.
Oracle provides cloud infrastructure and enterprise software solutions.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "Can you remind me what was the time when I asked about the current time?"})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Referring chat history.
The current time when you asked was 15:24:35 in 24-hour format, and 03:25:16 PM in 12-hour format.

> Finished chain.
Response: Referring chat history.
The current time when you asked was 15:24:35 in 24-hour format, and 03:25:16 PM in 12-hour format.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "What is the current time?"})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...

Invoking: `current_time` with `{}`


Current time: 15:27:48Not referring chat history.
The current time is 15:27:48.

> Finished chain.
Response: Not referring chat history.
The current time is 15:27:48.


In [ ]:
# Pretty print the chat history
print("\nChat History:")
for message in memory.load_memory_variables({}).get("chat_history"):
    print(message, "\n\n")


Chat History:
content='What is todays date?' additional_kwargs={} response_metadata={} 


content="Not referring chat history.\nToday's date is March 5, 2026." additional_kwargs={} response_metadata={} 


content='What is the current time?' additional_kwargs={} response_metadata={} 


content='Not referring chat history.\nThe current time is 15:24:35.' additional_kwargs={} response_metadata={} 


content='What is the current time in 12hr format?' additional_kwargs={} response_metadata={} 


content='Not referring chat history.\nThe current time in 12-hour format is 03:25:16 PM.' additional_kwargs={} response_metadata={} 


content='What is my name?' additional_kwargs={} response_metadata={} 


content="Not referring chat history.\nI'm sorry, but I don't have access to your name. Could you please tell me your name?" additional_kwargs={} response_metadata={} 


content='What is my system information?' additional_kwargs={} response_metadata={} 


content='Not referring chat history.\nYou

In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "Which all companies have we called so far?"})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Referring chat history.
So far, we've called information about the following companies:

1. Oracle

We attempted to get information about Walmart, but it was not available.

> Finished chain.
Response: Referring chat history.
So far, we've called information about the following companies:

1. Oracle

We attempted to get information about Walmart, but it was not available.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "do you have any information in your memory related to Walmart?"})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Referring chat history.
I attempted to retrieve information about Walmart earlier, but I was unable to provide any details at that time.

> Finished chain.
Response: Referring chat history.
I attempted to retrieve information about Walmart earlier, but I was unable to provide any details at that time.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "I work for a company called Walmart, which consumes cloud infrastructure and delivers cloud applications, Do you see any correlation with Oracle?"})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Not referring chat history.
Yes, there is a correlation between Walmart and Oracle in terms of cloud infrastructure and cloud applications. Oracle provides cloud infrastructure and enterprise software solutions, which can be utilized by companies like Walmart to support their cloud-based operations and applications. Walmart, as a consumer of cloud infrastructure, may use services from providers like Oracle to enhance their technological capabilities and deliver cloud applications effectively.

> Finished chain.
Response: Not referring chat history.
Yes, there is a correlation between Walmart and Oracle in terms of cloud infrastructure and cloud applications. Oracle provides cloud infrastructure and enterprise software solutions, which can be utilized by companies like Walmart to support their cloud-based operations and applications. Walmart, as a consumer of cloud infrastructure, may use services from providers like Oracle to enhance their techno

In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "What does OpenAI do?"})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...

Invoking: `get_company_info` with `{'company': 'OpenAI'}`


OpenAI builds advanced AI systems including ChatGPT and GPT models.Not referring chat history.
OpenAI builds advanced AI systems, including ChatGPT and GPT models.

> Finished chain.
Response: Not referring chat history.
OpenAI builds advanced AI systems, including ChatGPT and GPT models.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "Tell me about OpenAI."})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Not referring chat history.
OpenAI builds advanced AI systems, including ChatGPT and GPT models.

> Finished chain.
Response: Not referring chat history.
OpenAI builds advanced AI systems, including ChatGPT and GPT models.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "Define the features of MacBook M1 Pro."})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Not referring chat history.
I'm sorry, but I don't have information on the features of the MacBook M1 Pro at the moment.

> Finished chain.
Response: Not referring chat history.
I'm sorry, but I don't have information on the features of the MacBook M1 Pro at the moment.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "Define the features of MacBook M1 Air."})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Not referring chat history.
I'm sorry, but I don't have information on the features of the MacBook M1 Air at the moment.

> Finished chain.
Response: Not referring chat history.
I'm sorry, but I don't have information on the features of the MacBook M1 Air at the moment.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "Define the features of Macbook Intel processor laptops."})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Not referring chat history.
I'm sorry, but I don't have information on the features of MacBook laptops with Intel processors at the moment.

> Finished chain.
Response: Not referring chat history.
I'm sorry, but I don't have information on the features of MacBook laptops with Intel processors at the moment.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "What are the features of MacBook M1 Pro in json format."})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Not referring chat history.
I'm sorry, but I don't have information on the features of the MacBook M1 Pro at the moment.

> Finished chain.
Response: Not referring chat history.
I'm sorry, but I don't have information on the features of the MacBook M1 Pro at the moment.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "What does Oracle use to do in 1990s?"})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Not referring chat history.
In the 1990s, Oracle was primarily known for its database software. The company focused on developing and selling its flagship product, the Oracle Database, which became one of the most popular relational database management systems (RDBMS) in the industry. During this time, Oracle also expanded its product offerings to include enterprise software solutions, such as enterprise resource planning (ERP) and customer relationship management (CRM) systems, which helped businesses manage various aspects of their operations.

> Finished chain.
Response: Not referring chat history.
In the 1990s, Oracle was primarily known for its database software. The company focused on developing and selling its flagship product, the Oracle Database, which became one of the most popular relational database management systems (RDBMS) in the industry. During this time, Oracle also expanded its product offerings to include enterprise software s

In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "My name is Darshan, and I am struggling to get a job in Bangalore in some IT firm. Can you suggest some tips?"})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Not referring chat history.

Here are some tips to help you in your job search in Bangalore's IT sector:

1. **Update Your Resume**: Ensure your resume is up-to-date, highlighting relevant skills and experiences. Tailor it for each job application to match the job description.

2. **Enhance Your Skills**: Consider taking online courses or certifications in areas that are in demand, such as cloud computing, data science, AI, or software development.

3. **Networking**: Attend industry meetups, webinars, and conferences to connect with professionals. Networking can often lead to job opportunities.

4. **Leverage Job Portals**: Use job portals like LinkedIn, Naukri, and Indeed to search and apply for jobs. Set up alerts for new job postings that match your skills.

5. **Prepare for Interviews**: Practice common interview questions and technical questions related to your field. Mock interviews with friends or mentors can be helpful.

6. **Consider In

In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "What does Apple use to do in 1990s?"})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Not referring chat history.

In the 1990s, Apple was involved in several key activities and developments:

1. **Product Development**: Apple continued to innovate with its Macintosh line of computers. The company introduced products like the Macintosh LC, Macintosh Classic, and PowerBook laptops.

2. **Operating Systems**: Apple developed and released various versions of its Mac OS, enhancing the user experience and expanding its capabilities.

3. **Financial Struggles**: The early to mid-1990s were challenging for Apple financially, with declining market share and internal management struggles.

4. **Return of Steve Jobs**: In 1997, Steve Jobs returned to Apple after the company acquired NeXT, the company he founded after leaving Apple. His return marked a turning point for Apple.

5. **Revamping Product Lines**: Under Jobs' leadership, Apple streamlined its product lines and introduced the iMac in 1998, which was a commercial success and helped

In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "Do you recollect my name?"})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Referring chat history.
Yes, your name is Darshan.

> Finished chain.
Response: Referring chat history.
Yes, your name is Darshan.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "My name is Prashant, and I am looking for a sales job in Mumbai."})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Not referring chat history.

Here are some tips to help you in your search for a sales job in Mumbai:

1. **Update Your Resume**: Tailor your resume to highlight your sales achievements, skills, and experiences. Use specific metrics to showcase your success in previous roles.

2. **Leverage Online Platforms**: Utilize job portals like LinkedIn, Naukri, and Indeed to search for sales job opportunities in Mumbai. Set job alerts to stay updated with new postings.

3. **Networking**: Connect with professionals in the sales industry in Mumbai. Attend networking events, conferences, and seminars to meet potential employers and peers.

4. **Enhance Your Skills**: Consider taking courses or certifications in sales techniques, negotiation, or digital marketing to make yourself more attractive to employers.

5. **Prepare for Interviews**: Practice common sales interview questions and scenarios. Be ready to demonstrate your sales approach and how you handle

In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "what is the current time?"})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...

Invoking: `current_time` with `{}`


Current time: 15:36:38Not referring chat history.
The current time is 15:36:38.

> Finished chain.
Response: Not referring chat history.
The current time is 15:36:38.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "what is my CPU usage?"})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...

Invoking: `cpu_usage` with `{}`


CPU usage: 25.7%Not referring chat history.
Your current CPU usage is 25.7%.

> Finished chain.
Response: Not referring chat history.
Your current CPU usage is 25.7%.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "Do you recollect my name?"})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Referring chat history.
Yes, your name is Prashant.

> Finished chain.
Response: Referring chat history.
Yes, your name is Prashant.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "Do you recollect all names you have seen so far?"})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Referring chat history.
Yes, I have seen two names so far: Darshan and Prashant.

> Finished chain.
Response: Referring chat history.
Yes, I have seen two names so far: Darshan and Prashant.


In [ ]:
# Step 8: Invoke the agent
response = agent_executor.invoke({"input": "can you summarise the entire conversation in your memory so far?"})

# Step 9: Output the result
# print(response)
print(f"Response: {response['output']}")



> Entering new AgentExecutor chain...
Referring chat history.

Here's a summary of our conversation so far:

1. You asked for the current date and time, and I provided the information.
2. You inquired about your system information and battery percentage.
3. We discussed Oracle, and I provided a brief description of the company.
4. You mentioned working for Walmart and asked about its correlation with Oracle.
5. We talked about OpenAI and its activities.
6. You asked about the features of MacBook laptops with Intel processors and the MacBook M1 Pro, but I was unable to provide that information.
7. We discussed Oracle's activities in the 1990s.
8. You introduced yourself as Darshan, looking for a job in Bangalore's IT sector, and I provided some job search tips.
9. You later introduced yourself as Prashant, looking for a sales job in Mumbai, and I provided tips for that as well.
10. You asked about the current CPU usage, and I provided the information.

Throughout the conversation, I h

# Happy Learning